# End-to-End Machine Learning: Methods, Code, Outputs, and Production

**Interview preparation for experienced ML practitioners** | Companion to the local EDA methods cheat sheet and the end-to-end interview guide.

> **Importance key**  
> **[PRODUCTION CORE]** expected in most reliable ML systems.  
> **[PRODUCTION WHEN NEEDED]** use when justified by data, scale, risk, or architecture.  
> **[LEARNING / OPTIONAL]** useful method or optional dependency; not a default production requirement.

This notebook is deliberately broader than a single runnable pipeline. It demonstrates methods with small deterministic examples, prints representative outputs, and separates executable local examples from infrastructure templates that require your own services, credentials, data, or deployment environment.

## How to use

1. Run cells from top to bottom. Core examples use generated data and should not need downloads.
2. Read the production callout and leakage warning before adapting an example.
3. Optional-package cells print an installation hint when the dependency is not installed.
4. Infrastructure cells show templates and sample responses; Docker, Kubernetes, cloud, MLflow server, and CI/CD commands are not executed by this notebook.
5. Treat all generated outputs as demonstrations, not as evidence of business performance.

## Contents

Business framing; ingestion; EDA and statistics; validation and cleaning; missing data; outliers; splitting and validation; categorical encoding; scaling and transforms; feature engineering; selection and explainability; dimensionality reduction; regression and classification; clustering, anomaly detection and association rules; ensembles; tuning; evaluation; imbalance; diagnostics; production pipelines and leakage control; serialization; experiment/data versioning; API/batch serving; Docker/Kubernetes/CI-CD; monitoring/drift/retraining; registry/rollbacks; explainability; churn project blueprint; interview Q&A and study checklist.


## 0. Production rules to remember

**[PRODUCTION CORE]** Split before fitting any learned transform. Fit preprocessing, selection, and encoders only on training folds; keep them with the estimator in a pipeline. Use a final holdout set once. Match the split to the data (stratified, grouped, or temporal). Validate data at boundaries. Track code, data, parameters, metrics, and artifact versions. Never load untrusted pickle/joblib files.

Target leakage is information unavailable at prediction time. Train/test leakage includes fitting a scaler, imputer, selector, target encoder, or sampler on all rows before splitting. Time leakage uses future facts to predict the past. Group leakage puts records from the same entity in both training and evaluation when the deployment task requires unseen entities.

An interview answer should connect: **business decision -> prediction point -> available data -> split strategy -> baseline -> metric/cost -> validation -> deployment constraints -> monitoring and rollback.**


## 1. Business understanding and problem definition — [PRODUCTION CORE]

Before coding, agree on the unit of analysis, target definition, prediction timestamp, eligible population, decision/action, baseline, success metric, false-positive/false-negative costs, latency/throughput needs, privacy, fairness, explainability, and operating cost. Confirm every feature exists at the intended prediction time.


In [ ]:
problem_brief = {
    "decision": "Which eligible customers should receive a retention offer?",
    "unit_of_analysis": "one customer at a scoring date",
    "target": "customer cancels within the next 30 days",
    "prediction_time": "start of each day",
    "available_features": ["tenure_days", "support_tickets_30d", "monthly_fee"],
    "primary_metric": "recall at a reviewed-offer capacity of 500 customers/day",
    "baseline": "no-skill rate and current business rule",
    "constraints": ["daily batch by 06:00", "no post-cancellation fields"],
}
print("Target:", problem_brief["target"])
print("Prediction time:", problem_brief["prediction_time"])
print("Metric:", problem_brief["primary_metric"])
print("Leakage check: exclude features created after the scoring date.")

## 2. Data collection and ingestion — [PRODUCTION CORE]

Examples cover CSV, Excel, JSON, Parquet, SQL, and pickle. Pandas also supports `read_json`, `read_sql`, and `read_parquet`; files and connections are deliberately created in memory here. For APIs, object storage, Kafka, and production databases, add authentication, schema checks, retries/idempotency, source and ingestion-time metadata, and immutable raw-data retention. Only load pickle from a trusted source.


In [ ]:
from io import BytesIO, StringIO
import pickle
import sqlite3
import pandas as pd

csv_text = "id,age,city\n1,25,Paris\n2,40,Tokyo\n"
csv_df = pd.read_csv(StringIO(csv_text))
json_df = pd.read_json(StringIO('[{"id": 1, "age": 25}, {"id": 2, "age": 40}]'))
excel_buffer = BytesIO()
csv_df.to_excel(excel_buffer, index=False)
excel_buffer.seek(0)
excel_df = pd.read_excel(excel_buffer)
parquet_buffer = BytesIO()
csv_df.to_parquet(parquet_buffer, index=False)
parquet_buffer.seek(0)
parquet_df = pd.read_parquet(parquet_buffer)

with sqlite3.connect(":memory:") as connection:
    csv_df.to_sql("customers", connection, index=False)
    sql_df = pd.read_sql("SELECT id, age FROM customers WHERE age >= 30", connection)

pickle_buffer = BytesIO()
pickle.dump(csv_df, pickle_buffer)
pickle_buffer.seek(0)
pickle_df = pd.read_pickle(pickle_buffer)  # Only read trusted pickle files.

print("CSV / Excel / Parquet shapes:", csv_df.shape, excel_df.shape, parquet_df.shape)
print("JSON rows:", len(json_df), "| SQL result:", sql_df.to_dict("records"))
print("Pickle round trip:", pickle_df.equals(csv_df))
# File output APIs: csv_df.to_csv("clean.csv", index=False); csv_df.to_parquet("clean.parquet", index=False)
# File input APIs: pd.read_csv(path), pd.read_excel(path), pd.read_json(path),
#                  pd.read_parquet(path), pd.read_sql(query, connection), pd.read_pickle(trusted_path)

## 3. EDA: structure, quality, and column selection — [PRODUCTION CORE]

Inspect shape, memory, dtypes, examples, summaries, missingness, duplicate rows/IDs, category cardinality, and target distribution. EDA informs hypotheses; it does not prove causation. **Production:** convert repeated manual checks into schema/data-quality tests and monitoring.


In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    "age": [25, 30, 45, 50, 35, 42],
    "monthly_fee": [30.0, 45.0, 70.0, 55.0, 40.0, 85.0],
    "city": ["Paris", "Rome", "Paris", "Tokyo", "Rome", "Tokyo"],
    "plan": ["basic", "pro", "basic", "pro", "basic", "pro"],
    "churn": [0, 0, 1, 0, 0, 1],
})
print("shape / size / rows:", df.shape, df.size, len(df))
print("columns:", list(df.columns))
print("dtypes:\n", df.dtypes.to_dict())
print("numeric summary:\n", df.describe().round(2))
print("all-column summary:\n", df.describe(include="all").loc[["count", "unique", "top"]])
print("unique values:", df["city"].unique().tolist())
print("nunique:", df.nunique().to_dict())
print("frequencies:", df["city"].value_counts().to_dict())
print("duplicates:", int(df.duplicated().sum()))
print("deep memory bytes:", int(df.memory_usage(deep=True).sum()))
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
int_cols = df.select_dtypes(include=["int32", "int64"]).columns.tolist()
float_cols = df.select_dtypes(include=["float32", "float64"]).columns.tolist()
datetime_cols = df.select_dtypes(include=["datetime"]).columns.tolist()
print("numeric / categorical:", num_cols, cat_cols)
print("integer / float / datetime:", int_cols, float_cols, datetime_cols)
df.head(2), df.tail(2), df.sample(2, random_state=42)

## 4. EDA: numerical, categorical, visual, and statistical analysis — [PRODUCTION WHEN NEEDED]

Numerical summaries: mean, median, mode, min/max/range, variance, standard deviation, quantiles, skew, kurtosis. Categorical summaries: counts, proportions, unique levels, cardinality, crosstabs, grouped count/mean/aggregation. Relationships: Pearson/Spearman/Kendall correlations; scatter, hist/KDE, box/violin, ECDF, Q-Q, pair plots, count/bar/pie/stacked plots, and scatter matrices. Statistical tests below are examples; check assumptions and multiple-testing effects.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway, mannwhitneyu, ttest_ind, ttest_rel
from sklearn.feature_selection import mutual_info_classif

age = df["age"]
print("mean/median/mode:", age.mean(), age.median(), age.mode().tolist())
print("min/max/range:", age.min(), age.max(), age.max() - age.min())
print("variance/std:", round(age.var(), 2), round(age.std(), 2))
print("90th percentile / skew / kurtosis:", age.quantile(.9), round(age.skew(), 3), round(age.kurt(), 3))
print("city proportions:", df["city"].value_counts(normalize=True).round(2).to_dict())
print("cross-tab:\n", pd.crosstab(df["plan"], df["churn"]))
print("group size / mean:\n", df.groupby("plan").agg(count=("churn", "size"), churn_rate=("churn", "mean")))
print("correlations:\n", df.corr(numeric_only=True).round(2))
print("rank correlations:", df[["age", "monthly_fee"]].corr(method="spearman").iloc[0, 1],
      df[["age", "monthly_fee"]].corr(method="kendall").iloc[0, 1])
print("pearson r:", stats.pearsonr(df["age"], df["monthly_fee"]).statistic)
print("spearman rho:", stats.spearmanr(df["age"], df["monthly_fee"]).statistic)
print("MI feature scores:", mutual_info_classif(df[["age", "monthly_fee"]], df["churn"], random_state=42).round(3))

group_a, group_b = np.array([4.0, 5.0, 6.0, 5.0]), np.array([7.0, 8.0, 6.0, 9.0])
print("Welch independent t-test p:", round(ttest_ind(group_a, group_b, equal_var=False).pvalue, 4))
print("paired t-test p:", round(ttest_rel([3, 4, 5], [4, 5, 6]).pvalue, 4))
print("one-way ANOVA p:", round(f_oneway([1, 2, 3], [4, 5, 6], [7, 8, 9]).pvalue, 4))
print("Mann-Whitney p:", round(mannwhitneyu(group_a, group_b).pvalue, 4))
print("chi-square p:", round(chi2_contingency([[10, 4], [3, 9]]).pvalue, 4))
print("Q-Q plot inputs:", len(stats.probplot(age, dist="norm")[0][0]))
# Plot methods: sns.histplot(data=df, x="age", kde=True); sns.kdeplot(data=df, x="age")
# sns.boxplot(data=df, x="plan", y="monthly_fee"); sns.violinplot(data=df, x="plan", y="monthly_fee")
# sns.ecdfplot(data=df, x="age"); sns.countplot(data=df, x="plan", hue="churn")
# sns.pairplot(df.select_dtypes(include=np.number)); sns.heatmap(df.corr(numeric_only=True), annot=True)
# df["plan"].value_counts().plot(kind="bar"); .plot(kind="pie", autopct="%1.1f%%")
# pd.crosstab(df.plan, df.churn).plot(kind="bar", stacked=True)
# pd.plotting.scatter_matrix(df.select_dtypes(include=np.number)); plt.show()
plt.close("all")

## 5. Data validation and cleaning — [PRODUCTION CORE]

Validate schema, allowed values, ranges, nullability, unique IDs, types, and business rules. Clean duplicates, whitespace/case, spelling, punctuation, special characters, invalid numeric/date strings, and units. Review constant columns. A failed production check should alert, stop, or quarantine according to an explicit policy rather than silently accepting bad data.


In [ ]:
raw = pd.DataFrame({
    "customer_id": [1, 2, 2, 4],
    "city": [" Paris ", "ROME", "ROME", "Tokyo!"],
    "age": ["25", "40", "40", "bad"],
    "joined": ["2024-01-01", "2024-01-02", "2024-01-02", "not-a-date"],
    "status": ["active", "active", "active", "unknown"],
})
clean = raw.drop_duplicates().copy()
clean["city"] = clean["city"].str.strip().str.lower()
clean["city_no_punct"] = clean["city"].str.replace(r"[^\w\s]", "", regex=True)
clean["city_ascii_only"] = clean["city"].str.replace(r"[^a-zA-Z0-9 ]", "", regex=True)
clean["city"] = clean["city"].replace({"rome": "rome"})
clean["age"] = pd.to_numeric(clean["age"], errors="coerce")
clean["joined"] = pd.to_datetime(clean["joined"], errors="coerce")
clean["status"] = clean["status"].astype("category")
allowed_status = clean["status"].isin(["active", "inactive"])
age_in_range = clean["age"].between(0, 110)
duplicate_ids = clean["customer_id"].duplicated(keep=False)
constant_cols = [c for c in clean.columns if clean[c].nunique(dropna=False) <= 1]
clean = clean.rename(columns={"customer_id": "id"})
without_unused = clean.drop(columns=["city_ascii_only"])
print(clean[["id", "city", "age", "joined", "status"]].to_string(index=False))
print("valid status:", allowed_status.tolist(), "| age in range:", age_in_range.tolist())
print("duplicate IDs:", duplicate_ids.tolist(), "| constant columns:", constant_cols)
# Production schema libraries: Pandera DataFrameSchema/Column/Check/schema.validate(...)
# and Great Expectations expectations, suites, checkpoints, validation results, Data Docs.
# API examples require optional installs and project-specific schemas; do not make validation a silent default.

## 6. Missing-value detection, removal, and imputation — [PRODUCTION CORE]

Compare row/column removal (`dropna`, `axis=1`, `subset`) with pandas fixed/statistical fills and learned imputers. `SimpleImputer` strategies: mean, median, most_frequent, constant; `KNNImputer`; experimental `IterativeImputer`; `MissingIndicator`. Fit every learned imputer only on training folds. KNN/iterative methods need numeric, scaled/appropriately encoded inputs and can be costly.


In [ ]:
from sklearn.impute import KNNImputer, MissingIndicator, SimpleImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

missing = pd.DataFrame({
    "age": [20.0, np.nan, 40.0, 50.0, 30.0],
    "income": [30.0, 40.0, np.nan, 80.0, 50.0],
    "city": ["A", "B", None, "A", "B"],
})
print("null counts:", missing.isnull().sum().to_dict())
print("missing percentages:", (missing.isna().mean() * 100).round(1).to_dict())
print("non-null counts:", missing.notna().sum().to_dict())
print("drop rows/columns/subset shapes:", missing.dropna().shape, missing.dropna(axis=1).shape,
      missing.dropna(subset=["age"]).shape)
print("mean fill:", missing["age"].fillna(missing["age"].mean()).tolist())
print("median/mode fill:", missing["age"].fillna(missing["age"].median()).tolist(),
      missing["city"].fillna(missing["city"].mode()[0]).tolist())
print("ffill/bfill first rows:", missing.ffill().head(2).to_dict("records"),
      missing.bfill().head(2).to_dict("records"))
numeric = missing[["age", "income"]]
for strategy in ["mean", "median", "most_frequent", "constant"]:
    imp = SimpleImputer(strategy=strategy, fill_value=0 if strategy == "constant" else None)
    print("SimpleImputer", strategy, imp.fit_transform(numeric).round(2).tolist())
print("KNN:", KNNImputer(n_neighbors=2).fit_transform(numeric).round(2).tolist())
print("Iterative:", IterativeImputer(random_state=42, max_iter=10).fit_transform(numeric).round(2).tolist())
print("missing indicator:", MissingIndicator(features="all").fit_transform(numeric).astype(int).tolist())


## 7. Outlier detection and treatment — [PRODUCTION WHEN NEEDED]

Methods: IQR, Z-score, modified Z-score using MAD, percentile bounds, box plots, Isolation Forest, Local Outlier Factor, DBSCAN, Elliptic Envelope, and Winsorization. Options: investigate/correct errors, explicitly filter justified records, clip/cap, winsorize, transform valid positive skew (`log1p`), or use robust methods/scalers. Outlier flags are not proof of bad data; define and validate treatment using training data and domain review.


In [ ]:
from scipy.stats import median_abs_deviation, mstats, zscore
from sklearn.covariance import EllipticEnvelope
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import DBSCAN

values = pd.Series([10, 11, 12, 12, 13, 14, 15, 16, 100], name="value")
q1, q3 = values.quantile([.25, .75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
z_flags = np.abs(zscore(values)) > 3
mad = median_abs_deviation(values, scale="normal")
modified_z = .6745 * (values - values.median()) / median_abs_deviation(values)
percentile_bounds = values.quantile([.05, .95]).tolist()
print("IQR limits/outliers:", (lower, upper), values[(values < lower) | (values > upper)].tolist())
print("Z-score flags:", int(z_flags.sum()), "| MAD flagged:", values[np.abs(modified_z) > 3.5].tolist())
print("5th/95th percentiles:", percentile_bounds)
print("clipped:", values.clip(lower=lower, upper=upper).tolist())
print("winsorized:", mstats.winsorize(values, limits=(.1, .1)).tolist())
X_out = np.array([[10], [11], [12], [13], [14], [15], [100]], dtype=float)
print("IsolationForest:", IsolationForest(contamination=.15, random_state=42).fit_predict(X_out).tolist())
print("LOF:", LocalOutlierFactor(n_neighbors=3, contamination=.15).fit_predict(X_out).tolist())
print("DBSCAN:", DBSCAN(eps=3, min_samples=2).fit_predict(X_out).tolist())
print("EllipticEnvelope:", EllipticEnvelope(contamination=.15, random_state=42).fit_predict(X_out).tolist())
# Visualization: sns.boxplot(x=values); plt.show()
# Use np.log1p(x) only when x >= 0; robust scaling is an alternative when outliers are valid.


## 8. Train/validation/test splitting and cross-validation — [PRODUCTION CORE]

Use stratification for class balance, group splitters when entities repeat, and time splits for forecasting/temporal prediction. Available methods: `train_test_split`, `KFold`, `StratifiedKFold`, `RepeatedKFold`, `LeaveOneOut`, `GroupKFold`, `StratifiedGroupKFold`, `TimeSeriesSplit`, `cross_val_score`, and `cross_validate`. Every fitted preprocessing step must be inside the fold's pipeline.


In [ ]:
from sklearn.model_selection import (
    GroupKFold, KFold, LeaveOneOut, RepeatedKFold, StratifiedGroupKFold,
    StratifiedKFold, TimeSeriesSplit, cross_val_score, cross_validate, train_test_split,
)
from sklearn.linear_model import LogisticRegression

X = np.arange(120).reshape(60, 2)
y = np.array([0, 1] * 30)
groups = np.repeat(np.arange(20), 3)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.2, stratify=y, random_state=42
)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
model = LogisticRegression(max_iter=500)
print("holdout sizes/classes:", len(X_train), len(X_test), np.bincount(y_train).tolist())
print("KFold / Stratified folds:", KFold(3).get_n_splits(), cv.get_n_splits())
print("RepeatedKFold / LOO:", RepeatedKFold(n_splits=3, n_repeats=2, random_state=42).get_n_splits(),
      LeaveOneOut().get_n_splits(X))
print("Group / stratified group folds:", GroupKFold(3).get_n_splits(X, y, groups),
      StratifiedGroupKFold(3).get_n_splits(X, y, groups))
print("TimeSeriesSplit folds:", TimeSeriesSplit(n_splits=3).get_n_splits(X))
print("CV scores:", cross_val_score(model, X_train, y_train, cv=cv, scoring="f1").round(3).tolist())
print("cross_validate keys:", sorted(cross_validate(model, X_train, y_train, cv=cv,
      scoring=["accuracy", "f1"]).keys()))


## 9. Categorical encoding — [PRODUCTION CORE]

Nominal: one-hot/dummy; ordered: ordinal with explicit order. `map`/`replace` for fixed business mappings; category codes/factorize are arbitrary codes; count/frequency are compact but learned from training data; target/WOE/leave-one-out/CatBoost encoders are supervised and need leakage-safe out-of-fold fitting. `LabelEncoder` is for target labels, not ordinary nominal features. Unknown-category behavior must be deliberate.


In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder

cat = pd.DataFrame({"city": ["Paris", "London", "Paris", "Tokyo", "London"],
                    "size": ["low", "medium", "high", "medium", "low"],
                    "label": [0, 1, 0, 1, 0]})
mapping = {"Paris": 1, "London": 2, "Tokyo": 3}
cat["map"] = cat["city"].map(mapping)                 # unknown -> NaN
cat["replace"] = cat["city"].replace(mapping)         # unknown stays unchanged
cat["category_codes"] = cat["city"].astype("category").cat.codes
cat["factorize"], levels = pd.factorize(cat["city"])
counts = cat["city"].value_counts()
freq = cat["city"].value_counts(normalize=True)
cat["count"] = cat["city"].map(counts)
cat["frequency"] = cat["city"].map(freq)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(cat["label"])
onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe_array = onehot.fit_transform(cat[["city"]])
ohe_frame = pd.DataFrame(ohe_array, columns=onehot.get_feature_names_out(["city"]))
ordinal = OrdinalEncoder(categories=[["low", "medium", "high"]],
                         handle_unknown="use_encoded_value", unknown_value=-1)
cat["ordinal"] = ordinal.fit_transform(cat[["size"]]).ravel()
dummies = pd.get_dummies(cat["city"], prefix="city", drop_first=True, dtype=int)
print("map / replace:", cat[["map", "replace"]].iloc[:3].to_dict("list"))
print("levels / target classes:", levels.tolist(), label_encoder.classes_.tolist())
print("counts/frequencies:", cat[["count", "frequency"]].drop_duplicates().to_dict("records"))
print("one-hot columns:", onehot.get_feature_names_out(["city"]).tolist())
print("ordinal:", cat["ordinal"].tolist(), "| dummy shape:", dummies.shape)


### Optional categorical encoders — [PRODUCTION WHEN NEEDED]

**Optional:** `category-encoders` supports BinaryEncoder, HashingEncoder, CountEncoder, TargetEncoder, WOEEncoder, LeaveOneOutEncoder, CatBoostEncoder. Count/binary/hashing encoders do not use `y`; supervised encoders do. Scikit-learn `TargetEncoder` has cross-fitting support in current versions. Install only if selected; keep encoder fitting inside training folds/pipelines. **Output when unavailable:** a clear install hint.


In [ ]:
try:
    import category_encoders as ce
except ImportError:
    print("Optional examples skipped. Install with: python -m pip install category-encoders")
else:
    X_cat = cat[["city"]]
    y_cat = cat["label"]
    for name, encoder in {
        "binary": ce.BinaryEncoder(cols=["city"]),
        "hashing": ce.HashingEncoder(cols=["city"], n_components=4),
        "count": ce.CountEncoder(cols=["city"]),
        "target": ce.TargetEncoder(cols=["city"]),
        "WOE": ce.WOEEncoder(cols=["city"]),
        "leave-one-out": ce.LeaveOneOutEncoder(cols=["city"]),
        "CatBoost": ce.CatBoostEncoder(cols=["city"]),
    }.items():
        encoded = encoder.fit_transform(X_cat, y_cat) if name in {"target", "WOE", "leave-one-out", "CatBoost"} else encoder.fit_transform(X_cat)
        print(name, "shape:", encoded.shape)
# Current sklearn option: TargetEncoder(cv=5, shuffle=True, random_state=42).fit_transform(X_cat, y_cat)


## 10. Scaling and distribution transforms — [PRODUCTION WHEN NEEDED]

`StandardScaler`, `MinMaxScaler`, `RobustScaler`, `MaxAbsScaler`, `Normalizer` (per sample/vector), `QuantileTransformer`, and `PowerTransformer` (Yeo-Johnson or Box-Cox) are demonstrated. Scaling matters for linear models, SVM, KNN, PCA, and distance/gradient-based methods; trees often do not need it. Box-Cox requires strictly positive values. Fit only on training data.


In [ ]:
from sklearn.preprocessing import (
    MaxAbsScaler, MinMaxScaler, Normalizer, PowerTransformer,
    QuantileTransformer, RobustScaler, StandardScaler,
)

X_scale = np.array([[1.0, 100.0], [2.0, 300.0], [3.0, 500.0], [4.0, 1000.0]])
transformers = {
    "standard": StandardScaler(),
    "minmax": MinMaxScaler(),
    "robust": RobustScaler(),
    "maxabs": MaxAbsScaler(),
    "normalizer": Normalizer(),
    "quantile": QuantileTransformer(n_quantiles=4, output_distribution="normal", random_state=42),
    "yeo_johnson": PowerTransformer(method="yeo-johnson"),
    "box_cox": PowerTransformer(method="box-cox"),
}
for name, transformer in transformers.items():
    print(name, np.round(transformer.fit_transform(X_scale), 2).tolist())
# Fit: scaler.fit(X_train); transform: scaler.transform(X_valid_or_test); never fit on test/production batches.


## 11. Feature transformation and engineering — [PRODUCTION CORE]

Examples cover `log1p`, square root, cube root, reciprocal (guard zero), SciPy Box-Cox, Yeo-Johnson, polynomial/interactions, fixed/quantile bins (`cut`, `qcut`), date components, lag/rolling, group aggregations, string features, `CountVectorizer`, `TfidfVectorizer`, `assign`, `map`, `apply`, and `groupby().transform()`. Features must be available at scoring time. For time series, sort by time and use shift/rolling so no future observations enter past features.


In [ ]:
from scipy.stats import boxcox
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer, KBinsDiscretizer, PolynomialFeatures

feat = pd.DataFrame({
    "age": [18, 25, 42, 67], "amount": [0.0, 10.0, 50.0, 200.0],
    "customer": ["a", "a", "b", "b"], "date": pd.date_range("2024-01-01", periods=4),
    "text": ["great service", "fast delivery", "poor service", "great value"],
})
feat["log_amount"] = np.log1p(feat["amount"])
feat["sqrt_amount"] = np.sqrt(feat["amount"])
feat["cbrt_amount"] = np.cbrt(feat["amount"])
feat["reciprocal_amount"] = np.divide(1, feat["amount"], out=np.zeros(4), where=feat["amount"].to_numpy() != 0)
feat["boxcox_amount"], boxcox_lambda = boxcox(feat["amount"] + 1)  # shift makes values positive
feat["age_bin"] = pd.cut(feat["age"], bins=[0, 18, 35, 50, 70], include_lowest=True)
feat["age_quantile_bin"] = pd.qcut(feat["age"], q=2, labels=["younger", "older"])
feat["year"] = feat["date"].dt.year
feat["month"] = feat["date"].dt.month
feat["day"] = feat["date"].dt.day
feat["dayofweek"] = feat["date"].dt.dayofweek
feat["quarter"] = feat["date"].dt.quarter
feat["amount_lag1"] = feat["amount"].shift(1)
feat["amount_roll2_mean"] = feat["amount"].rolling(2, min_periods=1).mean()
feat["amount_roll2_std"] = feat["amount"].rolling(2, min_periods=2).std()
feat["customer_amount_mean"] = feat.groupby("customer")["amount"].transform("mean")
feat["age_squared"] = feat["age"].apply(lambda value: value ** 2)
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(feat[["age", "amount"]])
interactions = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False).fit_transform(feat[["age", "amount"]])
count = CountVectorizer()
tfidf = TfidfVectorizer()
count_matrix = count.fit_transform(feat["text"])
tfidf_matrix = tfidf.fit_transform(feat["text"])
custom = FunctionTransformer(np.log1p).fit_transform(feat[["amount"]])
bins = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="quantile").fit_transform(feat[["amount"]])
print("date features:", feat[["year", "month", "day", "dayofweek", "quarter"]].iloc[0].to_dict())
print("lag / rolling:", feat[["amount_lag1", "amount_roll2_mean", "amount_roll2_std"]].round(2).to_dict("records"))
print("group means:", feat["customer_amount_mean"].tolist(), "| polynomial shapes:", poly_features.shape, interactions.shape)
print("text vocabularies:", count.get_feature_names_out().tolist(), "| matrix shapes:", count_matrix.shape, tfidf_matrix.shape)
print("Box-Cox lambda / custom / bins:", round(boxcox_lambda, 3), custom.ravel().round(2).tolist(), bins.ravel().tolist())


## 12. Feature selection and model interpretation — [PRODUCTION WHEN NEEDED]

Filter: `VarianceThreshold`, `SelectKBest(f_classif|chi2)`, mutual information classification/regression, correlation. Wrapper: `RFE`, `RFECV`, forward/backward `SequentialFeatureSelector`. Embedded: Lasso, ElasticNet, `SelectFromModel(RandomForestClassifier)`, tree importances. Model-agnostic: `permutation_importance`; optional SHAP. Fit selection only within training folds. Importance indicates model association, not causality.


In [ ]:
from sklearn.feature_selection import (
    RFECV, RFE, SelectFromModel, SelectKBest, SequentialFeatureSelector,
    VarianceThreshold, f_classif, mutual_info_classif, mutual_info_regression, chi2,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.datasets import make_classification
X_sel, y_sel = make_classification(n_samples=80, n_features=6, n_informative=3,
                                    n_redundant=1, random_state=42)
print("variance filter:", VarianceThreshold().fit_transform(X_sel).shape)
for score in [f_classif, chi2, mutual_info_classif]:
    X_nonnegative = X_sel - X_sel.min(axis=0) if score is chi2 else X_sel
    print(score.__name__, SelectKBest(score_func=score, k=3).fit_transform(X_nonnegative, y_sel).shape)
print("MI regression scores:", mutual_info_regression(X_sel, y_sel, random_state=42).round(2).tolist())
print("correlation shape:", np.corrcoef(X_sel, rowvar=False).shape)
base = LogisticRegression(max_iter=500)
print("RFE / RFECV:", RFE(base, n_features_to_select=3).fit_transform(X_sel, y_sel).shape,
      RFECV(base, cv=3, min_features_to_select=2).fit_transform(X_sel, y_sel).shape)
print("sequential forward/backward:", SequentialFeatureSelector(base, n_features_to_select=3, direction="forward", cv=3).fit_transform(X_sel, y_sel).shape,
      SequentialFeatureSelector(base, n_features_to_select=3, direction="backward", cv=3).fit_transform(X_sel, y_sel).shape)
print("Lasso coefficients:", Lasso(alpha=.05, max_iter=5000).fit(X_sel, y_sel).coef_.round(2).tolist())
print("SelectFromModel:", SelectFromModel(RandomForestClassifier(n_estimators=20, random_state=42),
      threshold="median").fit_transform(X_sel, y_sel).shape)
forest = RandomForestClassifier(n_estimators=30, random_state=42).fit(X_sel, y_sel)
perm = permutation_importance(forest, X_sel, y_sel, n_repeats=3, random_state=42)
print("tree importances / permutation shape:", forest.feature_importances_.round(2).tolist(), perm.importances_mean.shape)
# Optional SHAP: shap.Explainer(model, background_data)(X); shap.plots.bar/beeswarm/waterfall.
try:
    import shap
except ImportError:
    print("Optional SHAP examples skipped. Install with: python -m pip install shap")


## 13. Dimensionality reduction — [PRODUCTION WHEN NEEDED]

PCA, TruncatedSVD, KernelPCA, supervised LDA, FastICA, t-SNE, Isomap, MDS, optional UMAP, and neural autoencoders are listed/demonstrated. Scale numeric features as appropriate. PCA/SVD create components (not original-feature selection). t-SNE is mainly visualization and is not a general-purpose out-of-sample transform. LDA is supervised and has class-count constraints. UMAP requires `umap-learn`; autoencoders require a deep-learning framework and problem-specific validation.


In [ ]:
from sklearn.decomposition import FastICA, KernelPCA, PCA, TruncatedSVD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import Isomap, MDS, TSNE
from sklearn.preprocessing import StandardScaler

X_dim, y_dim = make_classification(n_samples=40, n_features=6, n_informative=4,
                                    n_redundant=0, random_state=42)
X_scaled = StandardScaler().fit_transform(X_dim)
reducers = {
    "PCA": PCA(n_components=2),
    "TruncatedSVD": TruncatedSVD(n_components=2, random_state=42),
    "KernelPCA": KernelPCA(n_components=2, eigen_solver="arpack"),
    "FastICA": FastICA(n_components=2, random_state=42, max_iter=500, whiten="unit-variance"),
    "Isomap": Isomap(n_components=2, n_neighbors=5),
    "MDS": MDS(n_components=2, n_init=1, max_iter=30, random_state=42),
    "t-SNE": TSNE(n_components=2, perplexity=5, init="random", learning_rate="auto", random_state=42),
}
for name, reducer in reducers.items():
    print(name, reducer.fit_transform(X_scaled).shape)
print("LDA (supervised):", LinearDiscriminantAnalysis(n_components=1).fit_transform(X_scaled, y_dim).shape)
print("PCA explained variance:", PCA(n_components=2).fit(X_scaled).explained_variance_ratio_.round(3).tolist())
try:
    import umap
except ImportError:
    print("Optional UMAP skipped. Install with: python -m pip install umap-learn")
# Autoencoder: define an encoder/decoder network (e.g. TensorFlow/Keras), train to reconstruct
# training inputs, then validate the learned latent representation on held-out data.


## 14. Regression algorithms — [PRODUCTION CORE]

Included: LinearRegression, polynomial regression, Ridge, Lasso, ElasticNet, BayesianRidge, HuberRegressor, DecisionTree, RandomForest, ExtraTrees, GradientBoosting, AdaBoost, HistGradientBoosting, KNN, SVR, XGBoost, LightGBM, CatBoost. Compare a simple baseline with cross-validation; scale for scale-sensitive algorithms; tune and evaluate on an untouched holdout. External boosters are optional dependencies.


In [ ]:
from sklearn.datasets import make_regression
from sklearn.linear_model import BayesianRidge, ElasticNet, HuberRegressor, Lasso, LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    AdaBoostRegressor, ExtraTreesRegressor, GradientBoostingRegressor,
    HistGradientBoostingRegressor, RandomForestRegressor,
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

X_reg, y_reg = make_regression(n_samples=70, n_features=4, noise=10, random_state=42)
regressors = {
    "Linear": LinearRegression(), "Ridge": Ridge(), "Lasso": Lasso(alpha=.1),
    "ElasticNet": ElasticNet(alpha=.1), "BayesianRidge": BayesianRidge(),
    "Huber": HuberRegressor(), "DecisionTree": DecisionTreeRegressor(max_depth=3, random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=15, random_state=42),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=15, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=15, random_state=42),
    "AdaBoost": AdaBoostRegressor(n_estimators=15, random_state=42),
    "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=15, random_state=42),
    "KNN": KNeighborsRegressor(n_neighbors=3), "SVR": SVR(),
    "Polynomial+Linear": make_pipeline(PolynomialFeatures(degree=2), LinearRegression()),
}
for name, estimator in regressors.items():
    estimator.fit(X_reg, y_reg)
    print(name, "sample prediction:", round(float(estimator.predict(X_reg[:1])[0]), 2))
try:
    from xgboost import XGBRegressor
    from lightgbm import LGBMRegressor
    from catboost import CatBoostRegressor
    for name, estimator in {
        "XGBoost": XGBRegressor(n_estimators=10, max_depth=2, n_jobs=1, verbosity=0),
        "LightGBM": LGBMRegressor(n_estimators=10, max_depth=2, verbosity=-1),
        "CatBoost": CatBoostRegressor(iterations=10, depth=2, verbose=False, allow_writing_files=False),
    }.items():
        estimator.fit(X_reg, y_reg)
        print(name, "sample prediction:", round(float(estimator.predict(X_reg[:1])[0]), 2))
except ImportError as exc:
    print("Optional booster skipped:", exc)


## 15. Classification algorithms — [PRODUCTION CORE]

Included: LogisticRegression, SGDClassifier, Perceptron, KNN, Gaussian/Multinomial/Bernoulli Naive Bayes, DecisionTree, RandomForest, ExtraTrees, SVC/LinearSVC, LDA/QDA, AdaBoost, GradientBoosting, HistGradientBoosting, MLP, XGBoost, LightGBM, CatBoost. Scale for distance/margin/gradient methods; MultinomialNB expects nonnegative inputs. Compare cost-appropriate metrics and probability calibration where decisions need probabilities.


In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression, Perceptron, SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import BernoulliNB, GaussianNB, MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    AdaBoostClassifier, ExtraTreesClassifier, GradientBoostingClassifier,
    HistGradientBoostingClassifier, MLPClassifier, RandomForestClassifier,
)
from sklearn.svm import LinearSVC, SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

X_clf, y_clf = make_classification(n_samples=90, n_features=5, n_informative=3,
                                    n_redundant=0, random_state=42)
models = {
    "Logistic": LogisticRegression(max_iter=500), "SGD": SGDClassifier(random_state=42),
    "Perceptron": Perceptron(random_state=42), "KNN": KNeighborsClassifier(n_neighbors=3),
    "GaussianNB": GaussianNB(), "MultinomialNB": MultinomialNB(),
    "BernoulliNB": BernoulliNB(), "DecisionTree": DecisionTreeClassifier(max_depth=3, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=15, random_state=42),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=15, random_state=42),
    "SVC": SVC(probability=True, random_state=42), "LinearSVC": LinearSVC(random_state=42),
    "LDA": LinearDiscriminantAnalysis(), "QDA": QuadraticDiscriminantAnalysis(),
    "AdaBoost": AdaBoostClassifier(n_estimators=10, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=10, random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=10, random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(8,), max_iter=100, random_state=42),
}
X_nonnegative = X_clf - X_clf.min(axis=0)
X_binary = (X_clf > np.median(X_clf, axis=0)).astype(int)
for name, estimator in models.items():
    X_fit = X_nonnegative if name == "MultinomialNB" else X_binary if name == "BernoulliNB" else X_clf
    estimator.fit(X_fit, y_clf)
    print(name, "sample prediction:", int(estimator.predict(X_fit[:1])[0]))
try:
    from xgboost import XGBClassifier
    from lightgbm import LGBMClassifier
    from catboost import CatBoostClassifier
    for name, estimator in {
        "XGBoost": XGBClassifier(n_estimators=10, max_depth=2, n_jobs=1, eval_metric="logloss", verbosity=0),
        "LightGBM": LGBMClassifier(n_estimators=10, max_depth=2, verbosity=-1),
        "CatBoost": CatBoostClassifier(iterations=10, depth=2, verbose=False, allow_writing_files=False),
    }.items():
        estimator.fit(X_clf, y_clf)
        print(name, "sample prediction:", int(estimator.predict(X_clf[:1])[0]))
except ImportError as exc:
    print("Optional booster skipped:", exc)
# Shared estimator API: fit(X, y), predict(X), predict_proba(X) when supported,
# score(X, y), get_params(), set_params(...).

## 16. Unsupervised learning, clustering, anomaly detection, association rules — [PRODUCTION WHEN NEEDED]

Clustering: KMeans, Agglomerative, DBSCAN, HDBSCAN, OPTICS, MeanShift, GaussianMixture, SpectralClustering, Birch, AffinityPropagation. Anomaly detection: IsolationForest, LocalOutlierFactor, OneClassSVM, EllipticEnvelope. Association rules: mlxtend `apriori`, `fpgrowth`, `association_rules`. Scale distance-based inputs; DBSCAN/HDBSCAN labels `-1` denote noise. Evaluate clusters with domain usefulness as well as silhouette, Davies-Bouldin, Calinski-Harabasz; when known labels exist use ARI/NMI.


In [ ]:
from sklearn.cluster import (
    AffinityPropagation, AgglomerativeClustering, Birch, DBSCAN, KMeans,
    MeanShift, OPTICS, SpectralClustering,
)
from sklearn.mixture import GaussianMixture
from sklearn.svm import OneClassSVM
from sklearn.metrics import (
    adjusted_rand_score, calinski_harabasz_score, davies_bouldin_score,
    normalized_mutual_info_score, silhouette_score,
)

X_cluster = np.array([[0, 0], [0, 1], [1, 0], [8, 8], [8, 9], [9, 8]], dtype=float)
clusterers = {
    "KMeans": KMeans(n_clusters=2, n_init=10, random_state=42),
    "Agglomerative": AgglomerativeClustering(n_clusters=2),
    "DBSCAN": DBSCAN(eps=2, min_samples=2),
    "OPTICS": OPTICS(min_samples=2),
    "MeanShift": MeanShift(),
    "GaussianMixture": GaussianMixture(n_components=2, random_state=42),
    "Spectral": SpectralClustering(n_clusters=2, affinity="nearest_neighbors", random_state=42),
    "Birch": Birch(n_clusters=2),
    "AffinityPropagation": AffinityPropagation(random_state=42),
}
labels_by_name = {}
for name, estimator in clusterers.items():
    labels = estimator.fit_predict(X_cluster)
    labels_by_name[name] = labels
    print(name, labels.tolist())
labels = labels_by_name["KMeans"]
print("silhouette / Davies-Bouldin / Calinski-Harabasz:",
      round(silhouette_score(X_cluster, labels), 3),
      round(davies_bouldin_score(X_cluster, labels), 3),
      round(calinski_harabasz_score(X_cluster, labels), 2))
print("ARI / NMI (illustrative known labels):",
      round(adjusted_rand_score([0, 0, 0, 1, 1, 1], labels), 3),
      round(normalized_mutual_info_score([0, 0, 0, 1, 1, 1], labels), 3))
X_anomaly = np.array([[0, 0], [.1, 0], [0, .1], [.1, .1], [8, 8]], dtype=float)
print("IsolationForest:", IsolationForest(contamination=.2, random_state=42).fit_predict(X_anomaly).tolist())
print("LOF:", LocalOutlierFactor(n_neighbors=2, contamination=.2).fit_predict(X_anomaly).tolist())
print("OneClassSVM:", OneClassSVM(nu=.2).fit_predict(X_anomaly).tolist())
print("EllipticEnvelope:", EllipticEnvelope(contamination=.2, random_state=42).fit_predict(X_anomaly).tolist())
try:
    import hdbscan
except ImportError:
    print("Optional HDBSCAN skipped. Install with: python -m pip install hdbscan")
else:
    print("HDBSCAN:", hdbscan.HDBSCAN(min_cluster_size=2).fit_predict(X_cluster).tolist())
try:
    from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth
except ImportError:
    print("Optional association rules skipped. Install with: python -m pip install mlxtend")
else:
    baskets = pd.DataFrame({"bread": [1, 1, 0, 1], "milk": [1, 1, 1, 0], "eggs": [0, 1, 1, 1]}).astype(bool)
    itemsets = apriori(baskets, min_support=.5, use_colnames=True)
    print("apriori itemsets:", len(itemsets), "| fpgrowth itemsets:", len(fpgrowth(baskets, min_support=.5, use_colnames=True)))
    if len(itemsets):
        print("association rules:", len(association_rules(itemsets, metric="confidence", min_threshold=.5)))


## 17. Ensemble learning — [PRODUCTION WHEN NEEDED]

Bagging: BaggingClassifier/Regressor; Random Forest/Extra Trees. Boosting: AdaBoost, GradientBoosting, HistGradientBoosting, XGBoost, LightGBM, CatBoost. Combining: VotingClassifier/Regressor, StackingClassifier/Regressor. Bagging reduces variance through parallel resamples; boosting builds sequentially; voting combines predictions; stacking learns a meta-model using out-of-fold predictions. Keep validation leakage out of the stack.


In [ ]:
from sklearn.ensemble import (
    AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, HistGradientBoostingClassifier,
    RandomForestClassifier, StackingClassifier, VotingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification

X_ens, y_ens = make_classification(n_samples=90, n_features=5, n_informative=3,
                                    n_redundant=0, random_state=42)
base_estimators = [
    ("rf", RandomForestClassifier(n_estimators=10, random_state=42)),
    ("extra", ExtraTreesClassifier(n_estimators=10, random_state=42)),
    ("ada", AdaBoostClassifier(n_estimators=10, random_state=42)),
    ("gb", GradientBoostingClassifier(n_estimators=10, random_state=42)),
    ("hist", HistGradientBoostingClassifier(max_iter=10, random_state=42)),
]
ensembles = {
    "bagging": BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=10, random_state=42),
    "voting": VotingClassifier(estimators=base_estimators[:2] + [("lr", LogisticRegression(max_iter=500))]),
    "stacking": StackingClassifier(estimators=base_estimators[:2], final_estimator=LogisticRegression(max_iter=500), cv=3),
}
for name, estimator in ensembles.items():
    estimator.fit(X_ens, y_ens)
    print(name, estimator.predict(X_ens[:2]).tolist())
# Regressor counterparts: BaggingRegressor, RandomForestRegressor, AdaBoostRegressor,
# GradientBoostingRegressor, VotingRegressor, StackingRegressor, HistGradientBoostingRegressor.
# External boosting counterparts: XGB/LGBM/CatBoost Classifier and Regressor.


## 18. Hyperparameter tuning — [PRODUCTION CORE]

Grid search enumerates; randomized search samples; successive halving prunes weak candidates; Optuna/Hyperopt/Bayesian search adapt the next trial. Choose scoring for the business objective, use training-only CV, inspect `best_params_`, `best_score_`, `best_estimator_`, and assess on the test set once. Include preprocessing inside the estimator pipeline and prefix parameters (`model__max_depth`). Halving search may require scikit-learn's experimental enable import.


In [ ]:
from scipy.stats import randint
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import (
    GridSearchCV, HalvingGridSearchCV, HalvingRandomSearchCV,
    RandomizedSearchCV, StratifiedKFold, train_test_split,
)
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier

X_tune, y_tune = make_classification(n_samples=120, n_features=6, n_informative=4, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_tune, y_tune, stratify=y_tune, random_state=42)
cv_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
estimator = RandomForestClassifier(random_state=42, n_jobs=-1)
grid_params = {"n_estimators": [10, 20], "max_depth": [None, 3], "min_samples_leaf": [1, 2]}
searches = {
    "grid": GridSearchCV(estimator, grid_params, cv=cv_tune, scoring="f1", n_jobs=-1),
    "random": RandomizedSearchCV(estimator, {
        "n_estimators": randint(10, 31), "max_depth": [None, 3, 5],
        "min_samples_leaf": randint(1, 4)}, n_iter=4, cv=cv_tune,
        scoring="f1", random_state=42, n_jobs=-1),
    "halving_grid": HalvingGridSearchCV(estimator, grid_params, cv=cv_tune, scoring="f1", factor=2, n_jobs=-1),
    "halving_random": HalvingRandomSearchCV(estimator, {
        "n_estimators": randint(10, 31), "max_depth": [None, 3, 5]},
        n_candidates=4, cv=cv_tune, scoring="f1", random_state=42, factor=2, n_jobs=-1),
}
for name, search in searches.items():
    search.fit(X_tr, y_tr)
    print(name, "best params:", search.best_params_, "| CV F1:", round(search.best_score_, 3),
          "| holdout F1:", round(search.score(X_te, y_te), 3))
# Optional Optuna: create_study(direction="maximize"), trial.suggest_int/float/categorical,
# study.optimize(objective, n_trials=...), study.best_params.
# Optional scikit-optimize: BayesSearchCV(estimator, search_spaces, n_iter=..., cv=...).
# Optional Hyperopt: fmin(objective, space=..., algo=tpe.suggest, max_evals=...).
try:
    import optuna
except ImportError:
    print("Optional Optuna skipped. Install with: python -m pip install optuna")
try:
    from skopt import BayesSearchCV
except ImportError:
    print("Optional BayesSearchCV skipped. Install with: python -m pip install scikit-optimize")
try:
    from hyperopt import fmin, tpe
except ImportError:
    print("Optional Hyperopt skipped. Install with: python -m pip install hyperopt")


## 19. Model evaluation — [PRODUCTION CORE]

Classification: accuracy, precision, recall, F1, confusion matrix/report, ROC-AUC/curve, PR curve/AP, log loss, MCC, Cohen kappa, balanced accuracy. Regression: MAE, MSE, RMSE, R2, explained variance, MAPE (unstable near zero), median absolute error, MSLE (nonnegative domain). Clustering: silhouette, Davies-Bouldin, Calinski-Harabasz, ARI/NMI when reference labels exist. Always report the averaging/positive-label convention for multiclass/imbalanced metrics.


In [ ]:
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    cohen_kappa_score, classification_report, confusion_matrix,
    explained_variance_score, f1_score, log_loss, matthews_corrcoef,
    mean_absolute_error, mean_absolute_percentage_error, mean_squared_error,
    median_absolute_error, precision_recall_curve, precision_score,
    r2_score, recall_score, roc_auc_score, roc_curve,
)

y_true = np.array([0, 1, 1, 0, 1, 0])
y_pred = np.array([0, 1, 0, 0, 1, 1])
y_prob = np.array([.1, .8, .6, .2, .9, .7])
print("classification:", {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1": f1_score(y_true, y_pred, zero_division=0),
    "roc_auc": round(roc_auc_score(y_true, y_prob), 3),
    "average_precision": round(average_precision_score(y_true, y_prob), 3),
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    "MCC": round(matthews_corrcoef(y_true, y_pred), 3),
    "kappa": round(cohen_kappa_score(y_true, y_pred), 3),
    "log_loss": round(log_loss(y_true, y_prob), 3),
})
print("confusion matrix:\n", confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, zero_division=0))
fpr, tpr, roc_thresholds = roc_curve(y_true, y_prob)
pr, rc, pr_thresholds = precision_recall_curve(y_true, y_prob)
print("ROC/PR curve point counts:", len(fpr), len(pr))
y_reg_true = np.array([3., -0.5, 2., 7.])
y_reg_pred = np.array([2.5, 0., 2., 8.])
mse = mean_squared_error(y_reg_true, y_reg_pred)
print("regression:", {
    "MAE": mean_absolute_error(y_reg_true, y_reg_pred),
    "MSE": mse, "RMSE": mse ** .5, "R2": round(r2_score(y_reg_true, y_reg_pred), 3),
    "explained_variance": round(explained_variance_score(y_reg_true, y_reg_pred), 3),
    "MAPE": round(mean_absolute_percentage_error(y_reg_true, y_reg_pred), 3),
    "median_absolute_error": median_absolute_error(y_reg_true, y_reg_pred),
})


## 20. Class imbalance and decision thresholds — [PRODUCTION WHEN NEEDED]

Options: `class_weight="balanced"`, RandomOverSampler, RandomUnderSampler, SMOTE, SMOTETomek, SMOTEENN, ADASYN, and threshold selection on validation predictions. Resample **only within training folds** using `imblearn.pipeline.Pipeline`; never resample validation/test data. Check minority sample count and feature type before synthetic sampling.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score

y_imb = np.array([0] * 18 + [1] * 2)
X_imb = np.arange(40).reshape(20, 2)
weighted = LogisticRegression(class_weight="balanced", max_iter=500).fit(X_imb, y_imb)
probabilities = weighted.predict_proba(X_imb)[:, 1]
threshold = .35  # choose on validation data using business costs/capacity
pred_threshold = (probabilities >= threshold).astype(int)
print("class counts:", np.bincount(y_imb).tolist())
print("threshold / precision / recall:", threshold,
      round(precision_score(y_imb, pred_threshold, zero_division=0), 3),
      round(recall_score(y_imb, pred_threshold, zero_division=0), 3))
try:
    from imblearn.over_sampling import ADASYN, RandomOverSampler, SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    from imblearn.combine import SMOTEENN, SMOTETomek
except ImportError:
    print("Optional samplers skipped. Install with: python -m pip install imbalanced-learn")
else:
    from sklearn.datasets import make_classification
    X_s, y_s = make_classification(n_samples=100, n_features=4, weights=[.9, .1], random_state=42)
    samplers = {
        "RandomOverSampler": RandomOverSampler(random_state=42),
        "RandomUnderSampler": RandomUnderSampler(random_state=42),
        "SMOTE": SMOTE(random_state=42),
        "SMOTETomek": SMOTETomek(random_state=42),
        "SMOTEENN": SMOTEENN(random_state=42),
        "ADASYN": ADASYN(random_state=42),
    }
    for name, sampler in samplers.items():
        try:
            _, y_res = sampler.fit_resample(X_s, y_s)
            print(name, np.bincount(y_res).tolist())
        except ValueError as exc:
            print(name, "not applicable to this small example:", exc)
# CV-safe pattern: imblearn.pipeline.Pipeline([("scale", StandardScaler()),
# ("sampler", SMOTE()), ("model", LogisticRegression())])


## 21. Diagnostics and error analysis — [PRODUCTION CORE]

Inspect residuals (`y_true - y_pred`), residual-vs-prediction plots, learning curves, validation curves, probability calibration curves, threshold behavior, group-segmented errors, bias/variance, over/underfit, and feature importance. Segment only on legitimate, sufficiently sized groups and follow privacy/fairness policies.


In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.model_selection import learning_curve, validation_curve
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

X_diag, y_diag = make_classification(n_samples=90, n_features=5, n_informative=3, random_state=42)
diag_model = LogisticRegression(max_iter=500)
diag_model.fit(X_diag[:70], y_diag[:70])
pred_prob = diag_model.predict_proba(X_diag[70:])[:, 1]
pred_label = (pred_prob >= .5).astype(int)
residuals = y_diag[70:] - pred_label
print("residual examples:", residuals[:5].tolist())
print("error rate by group:", pd.DataFrame({"group": np.where(X_diag[70:, 0] > 0, "positive_x", "negative_x"),
      "error": (pred_label != y_diag[70:]).astype(int)}).groupby("group")["error"].mean().round(3).to_dict())
print("calibration bins:", [len(x) for x in calibration_curve(y_diag[70:], pred_prob, n_bins=3)])
train_sizes, train_scores, valid_scores = learning_curve(
    LogisticRegression(max_iter=500), X_diag, y_diag, cv=3, train_sizes=[.5, 1.0])
print("learning curve shapes:", train_sizes.tolist(), train_scores.shape, valid_scores.shape)
param_range = [0.01, 0.1, 1.0]
_, val_train, val_test = validation_curve(
    LogisticRegression(max_iter=500), X_diag, y_diag, param_name="C",
    param_range=param_range, cv=3, scoring="accuracy")
print("validation curve:", val_train.shape, val_test.shape)
# Visuals: sns.residplot(x=y_pred, y=y_true-y_pred); calibration_curve(...); plt.show()


## 22. Preprocessing pipelines and leakage-safe end-to-end training — [PRODUCTION CORE]

Use `Pipeline`, `ColumnTransformer`, `make_pipeline`, and `make_column_transformer` to apply separate numeric/categorical preprocessing and estimator fitting. Include sampling via `imblearn.pipeline.Pipeline` when needed. This makes cross-validation fit transforms separately in each fold and ensures the same transformations are used in serving.


In [ ]:
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

df_pipe = pd.DataFrame({
    "age": [22, 25, 28, 35, 40, 45, 50, 55, 60, 65],
    "salary": [30, 40, 45, 55, 60, 70, 75, 80, 90, 100],
    "city": ["A", "B", "A", "C", "B", "A", "C", "B", "A", "C"],
    "target": [0, 0, 0, 0, 1, 1, 1, 1, 1, 1],
})
X_p, y_p = df_pipe.drop(columns="target"), df_pipe["target"]
Xtr, Xte, ytr, yte = train_test_split(X_p, y_p, test_size=.3, stratify=y_p, random_state=42)
numeric = X_p.select_dtypes(include=np.number).columns
categorical = X_p.select_dtypes(include=["object", "category"]).columns
numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                             ("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("num", numeric_pipe, numeric), ("cat", categorical_pipe, categorical)])
full_pipeline = Pipeline([("preprocessor", preprocessor),
                          ("model", RandomForestClassifier(n_estimators=20, random_state=42))])
full_pipeline.fit(Xtr, ytr)
pred = full_pipeline.predict(Xte)
print("predictions:", pred.tolist())
print(classification_report(yte, pred, zero_division=0))
print("named steps:", list(full_pipeline.named_steps))
simple_pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500))
simple_pipeline.fit(np.asarray(Xtr[numeric]), ytr)
print("make_pipeline prediction:", simple_pipeline.predict(np.asarray(Xte[numeric])).tolist())
column_helper = make_column_transformer((StandardScaler(), list(numeric)),
                                        (OneHotEncoder(handle_unknown="ignore"), list(categorical)))
print("make_column_transformer output shape:", column_helper.fit_transform(Xtr).shape)
# Tuning parameters use names such as preprocessor__num__imputer__strategy / model__max_depth.


## 23. Serialization, reproducibility, and artifact safety — [PRODUCTION CORE]

Save the **complete fitted preprocessing-plus-model pipeline** with `joblib.dump/load` or `pickle.dump/load`; record Python/library versions, feature schema, training data version, code commit, parameters, metrics, and artifact hash. Pickle/joblib can execute code when loading: load only trusted artifacts. Prefer versioned, access-controlled storage.


In [ ]:
import hashlib
import joblib
import pickle
import tempfile
from pathlib import Path

sample = Xte.iloc[:2]
with tempfile.TemporaryDirectory() as tmp:
    artifact = Path(tmp) / "pipeline.joblib"
    joblib.dump(full_pipeline, artifact)
    restored = joblib.load(artifact)
    print("joblib identical predictions:", restored.predict(sample).tolist() == full_pipeline.predict(sample).tolist())
    digest = hashlib.sha256(artifact.read_bytes()).hexdigest()
    print("artifact sha256 prefix:", digest[:12])
    pkl = Path(tmp) / "pipeline.pkl"
    with pkl.open("wb") as stream:
        pickle.dump(full_pipeline, stream)
    with pkl.open("rb") as stream:
        restored_pickle = pickle.load(stream)  # trusted artifact only
    print("pickle prediction:", restored_pickle.predict(sample).tolist())
print("python:", __import__("platform").python_version(), "| sklearn:", __import__("sklearn").__version__)


## 24. Experiment tracking with MLflow — [PRODUCTION WHEN NEEDED]

Methods: `set_tracking_uri`, `set_experiment`, `start_run`, `log_param(s)`, `log_metric(s)`, `log_artifact`, `mlflow.sklearn.log_model`, `register_model`, `mlflow.pyfunc.load_model`. Run the template in an environment configured with an MLflow tracking/artifact store. Configure authentication, access control, durable storage, backups, and model approval in production.


In [ ]:
# Template: run after setting up a tracking server and trusted artifact store.
# import mlflow
# import mlflow.sklearn
# mlflow.set_tracking_uri("http://127.0.0.1:5000")
# mlflow.set_experiment("customer_churn")
# with mlflow.start_run():
#     mlflow.log_params({"model": "RandomForestClassifier", "n_estimators": 100})
#     mlflow.log_metrics({"f1": 0.81, "recall": 0.87})
#     mlflow.log_artifact("data_schema.json")
#     mlflow.sklearn.log_model(full_pipeline, name="model")
# registered = mlflow.register_model(model_uri="runs:/<run-id>/model", name="customer-churn")
# loaded = mlflow.pyfunc.load_model(f"models:/customer-churn/{registered.version}")
print("Expected tracking result: a run containing parameters, metrics, artifacts, and a versioned model.")
print("Start local server (example): mlflow server --host 127.0.0.1 --port 5000")


## 25. Data and pipeline versioning with DVC — [PRODUCTION WHEN NEEDED]

Typical commands: `dvc init`, `dvc add data/train.csv`, `dvc remote add -d storage <uri>`, `dvc push`, `dvc pull`, `dvc status`, `dvc repro`, `dvc dag`. Git versions code and small DVC pointer files; DVC remotes store large data/artifacts. Protect remote access and credentials.


In [ ]:
# Run these in an explicitly initialized project/repository; they are not executed here.
# dvc init
# dvc add data/train.csv
# git add data/train.csv.dvc .gitignore
# dvc remote add -d storage s3://my-bucket/ml-data
# dvc push
# dvc status
# dvc pull
# dvc repro
# dvc dag
print("Expected workflow: Git records code/metadata; DVC tracks data and reproducible pipeline artifacts.")


## 26. FastAPI model serving — [PRODUCTION WHEN NEEDED]

FastAPI app, Pydantic request validation, GET health endpoint, POST prediction endpoint, `model.predict`, optional `predict_proba`, `HTTPException`, and Uvicorn. Load the fitted pipeline once at application startup. Add authentication/authorization, rate and payload limits, timeouts, structured logs/request IDs, health/readiness checks, versioned schema, and controlled errors before production.


In [ ]:
# Save as app.py after replacing artifact path and request fields for your pipeline.
# from contextlib import asynccontextmanager
# import joblib
# import pandas as pd
# from fastapi import FastAPI, HTTPException
# from pydantic import BaseModel
#
# class Customer(BaseModel):
#     age: int
#     salary: float
#     city: str
#
# @asynccontextmanager
# async def lifespan(app):
#     app.state.model = joblib.load("model.joblib")  # load trusted artifact once
#     yield
#
# app = FastAPI(lifespan=lifespan)
#
# @app.get("/health")
# def health():
#     return {"status": "ok"}
#
# @app.post("/predict")
# def predict(customer: Customer):
#     row = pd.DataFrame([customer.model_dump()])
#     try:
#         prediction = app.state.model.predict(row)
#         response = {"prediction": prediction.tolist()}
#         if hasattr(app.state.model, "predict_proba"):
#             response["probability"] = app.state.model.predict_proba(row).tolist()
#         return response
#     except ValueError as exc:
#         raise HTTPException(status_code=422, detail="Invalid model input") from exc
#
# Run: uvicorn app:app --host 0.0.0.0 --port 8000
print('GET /health -> {"status": "ok"}')
print('POST /predict -> {"prediction": [1], "probability": [[0.12, 0.88]]} (illustrative)')
print("Interactive API docs: http://127.0.0.1:8000/docs")


## 27. Batch and real-time inference — [PRODUCTION WHEN NEEDED]

Batch: periodically score chunks (`read_csv(chunksize=...)`, `predict(batch)`), write versioned results, orchestrate with a scheduler/workflow engine (e.g. Airflow). Real-time: gateway -> API -> schema validation -> same fitted preprocessing/model -> response and structured telemetry. Track throughput, p50/p95/p99 latency, errors, resource use, and idempotency; use queues/caching/rate limiting only when justified.


In [ ]:
# In-memory batch example; chunk the source rather than loading a huge file at once.
new_rows = Xte.copy()
batch_predictions = full_pipeline.predict(new_rows)
batch_result = new_rows.assign(prediction=batch_predictions)
print("batch rows/predictions:", len(batch_result), batch_result["prediction"].tolist())
# Real CSV pattern:
# for chunk in pd.read_csv("incoming.csv", chunksize=10_000):
#     scored = full_pipeline.predict(chunk)
#     write_scored_batch(scored, model_version="v1")
# Real-time response is illustrated in the FastAPI cell; measure latency and error rate around predict().


## 28. Docker: package the model service — [PRODUCTION WHEN NEEDED]

Pin dependencies, use a maintained small base image, run as non-root, keep secrets out of the image, add health checks, scan images, and use immutable tags. Commands: build, images, run, ps, stop, logs, exec, pull, push, Docker Compose. Registry examples: Docker Hub, AWS ECR, Azure Container Registry, Google Artifact Registry.


In [ ]:
# Dockerfile template (requires app.py, model.joblib, and pinned requirements.txt)
# FROM python:3.12-slim
# WORKDIR /app
# COPY requirements.txt .
# RUN pip install --no-cache-dir -r requirements.txt
# COPY app.py model.joblib ./
# USER 10001
# EXPOSE 8000
# HEALTHCHECK CMD python -c "import urllib.request; urllib.request.urlopen('http://127.0.0.1:8000/health')"
# CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
#
# Commands (execute from the service project directory):
# docker build -t churn-api:v1 .
# docker images
# docker run -d --name churn-api -p 8000:8000 churn-api:v1
# docker ps
# docker logs churn-api
# docker exec -it churn-api sh
# docker stop churn-api
# docker pull registry.example/churn-api:v1
# docker push registry.example/churn-api:v1
# docker compose up -d
print("Expected: image churn-api:v1 built; health endpoint available on localhost:8000.")


## 29. Kubernetes deployment and scaling — [PRODUCTION WHEN NEEDED]

Objects: Pod, Deployment, Service, optional Ingress, ConfigMap, Secret, HPA, Namespace, PersistentVolume/Claim. Set immutable image tags, selectors, resource requests/limits, readiness/liveness probes, configuration references, and least privilege. Ingress is only needed for external HTTP routing/TLS. Validate YAML and cluster policies before apply.


In [ ]:
# deployment.yaml template (replace image and resource values for your environment)
# apiVersion: apps/v1
# kind: Deployment
# metadata: {name: ml-api}
# spec:
#   replicas: 2
#   selector: {matchLabels: {app: ml-api}}
#   template:
#     metadata: {labels: {app: ml-api}}
#     spec:
#       containers:
#         - name: api
#           image: registry.example/churn-api:v1
#           ports: [{containerPort: 8000}]
#           resources:
#             requests: {cpu: "100m", memory: "256Mi"}
#             limits: {cpu: "500m", memory: "512Mi"}
#           readinessProbe: {httpGet: {path: /health, port: 8000}}
#           livenessProbe: {httpGet: {path: /health, port: 8000}}
# ---
# apiVersion: v1
# kind: Service
# metadata: {name: ml-api}
# spec:
#   selector: {app: ml-api}
#   ports: [{port: 80, targetPort: 8000}]
#   type: ClusterIP
#
# Commands: kubectl get pods/deployments/services; kubectl apply -f deployment.yaml;
# kubectl describe pod <pod>; kubectl logs <pod>; kubectl scale deployment/ml-api --replicas=3;
# kubectl rollout status deployment/ml-api; kubectl rollout history deployment/ml-api;
# kubectl rollout undo deployment/ml-api
print("Expected: Deployment maintains replicas; Service gives stable in-cluster access; probes gate traffic.")


## 30. CI/CD for machine learning — [PRODUCTION CORE]

CI stages: checkout, install locked dependencies, format/lint, unit/integration tests, schema/data validation, model tests, reproducibility/metric thresholds, fairness/bias checks as applicable, dependency/container security scan, build artifact/image. CD: publish immutable artifact, deploy staging, smoke/integration tests, gate approval, promote safely, monitor, rollback. GitHub Actions uses `on`, `jobs`, `steps`, `uses`, `run`, `env`, secrets and artifacts; Jenkins uses Jenkinsfile/pipeline/agent/stages/steps/post/credentials/shared libraries.


In [ ]:
# .github/workflows/ml-ci.yml skeleton (adapt commands and protect secrets):
# name: ml-ci
# on: [push, pull_request]
# jobs:
#   test-and-build:
#     runs-on: ubuntu-latest
#     steps:
#       - uses: actions/checkout@v4
#       - uses: actions/setup-python@v5
#         with: {python-version: "3.12"}
#       - run: pip install -r requirements.txt
#       - run: python -m pytest
#       - run: python -m ruff check .
#       - run: python validate_data.py
#       - run: python evaluate.py --minimum-f1 0.80
#       - run: docker build -t ml-api:${{ github.sha }} .
# Jenkins concepts: Jenkinsfile -> pipeline { agent; stages { stage { steps { sh(...) } } }; post {...} }
print("CI gate example: tests pass AND schema valid AND metric threshold met before image promotion.")


## 31. Monitoring, data quality, drift, and retraining — [PRODUCTION CORE]

Monitor data quality (missing/invalid/schema/duplicates/categories/volume), feature and prediction drift, model quality once labels arrive, and service health (volume, p50/p95/p99 latency, error rate, CPU/memory, availability). Tools: Evidently, Prometheus, Grafana, OpenTelemetry, ELK/OpenSearch, CloudWatch/Azure Monitor. Data drift is input-distribution change; concept drift is a change in P(y|X), usually confirmed with outcomes. Drift is a signal, not an automatic retraining trigger. Evaluate business significance and labels; retrain candidate, validate fairness/business/technical gates, compare to champion, stage, promote, monitor, preserve rollback.


In [ ]:
from scipy.stats import ks_2samp, wasserstein_distance

reference = np.array([10, 11, 12, 13, 14, 15, 16, 17], dtype=float)
current = np.array([10, 12, 13, 15, 17, 18, 20, 22], dtype=float)
ks = ks_2samp(reference, current)
wd = wasserstein_distance(reference, current)
def population_stability_index(expected, actual, bins=5):
    edges = np.unique(np.quantile(np.r_[expected, actual], np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    e = np.histogram(expected, bins=edges)[0] / len(expected)
    a = np.histogram(actual, bins=edges)[0] / len(actual)
    eps = 1e-6
    e, a = np.clip(e, eps, 1), np.clip(a, eps, 1)
    return float(np.sum((a - e) * np.log(a / e)))
print("KS statistic/p:", round(ks.statistic, 3), round(ks.pvalue, 3))
print("Wasserstein distance / PSI:", round(wd, 3), round(population_stability_index(reference, current), 3))
print("Data quality:", {"missing_rate": 0.01, "invalid_rate": 0.0, "schema_ok": True})
print("Service metrics:", {"requests_per_min": 120, "p95_latency_ms": 45, "error_rate": 0.002})
# Optional Evidently: Report(metrics=[DataDriftPreset()]).run(reference_data=ref_df, current_data=cur_df)
# Prometheus exposes metrics; Grafana alerts/dashboards; OpenTelemetry traces/metrics/logs.


## 32. Registry, champion/challenger, safe rollout, rollback — [PRODUCTION CORE]

Version model, training data, code commit, feature definitions, environment, parameters, metrics, approvals, deployment environment, and artifact URI. A challenger must meet identical holdout/business/fairness/latency criteria before promotion. Use shadow/canary/A-B testing where appropriate; define rollback triggers before release and keep the previous artifact/image/deployment.


In [ ]:
candidate_metrics = {"champion_recall": .82, "challenger_recall": .86,
                     "champion_p95_ms": 40, "challenger_p95_ms": 43}
min_recall = .80
max_latency_ms = 60
promote = (candidate_metrics["challenger_recall"] >= min_recall
           and candidate_metrics["challenger_p95_ms"] <= max_latency_ms)
registry_record = {
    "model_name": "customer-churn", "version": "2",
    "data_version": "dvc:abc123", "code_commit": "git:deafbeef",
    "status": "candidate", "metrics": candidate_metrics,
}
print("promote challenger:", promote)
print("registry fields:", sorted(registry_record))
print("rollback command example: kubectl rollout undo deployment/ml-api")
# A/B test: randomize eligible traffic, log assignment and outcomes, compare predeclared metrics.
# Shadow: score a copy of traffic without affecting decisions; compare outputs and latency.
# Canary: route a small fraction, monitor SLOs, then expand or roll back.


## 33. Model explainability — [PRODUCTION WHEN NEEDED]

SHAP: `shap.Explainer(model, background_data)`, explainer(X), global `shap.plots.bar` / beeswarm and local waterfall. LIME: `LimeTabularExplainer(...).explain_instance(row, predict_fn).show_in_notebook()`. Explainability can support debugging and communication; feature importance is not causality. Validate explanations for the fitted pipeline/model and disclose limitations.


In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

X_explain, y_explain = make_classification(n_samples=80, n_features=5, n_informative=3, random_state=42)
explain_model = RandomForestClassifier(n_estimators=30, random_state=42).fit(X_explain, y_explain)
importance = permutation_importance(explain_model, X_explain, y_explain, n_repeats=3, random_state=42)
print("permutation importance:", importance.importances_mean.round(3).tolist())
try:
    import shap
    explainer = shap.Explainer(explain_model, X_explain[:20])
    explanation = explainer(X_explain[:3])
    print("SHAP values shape:", explanation.values.shape)
    # shap.plots.bar(explanation); shap.plots.beeswarm(explanation); shap.plots.waterfall(explanation[0])
except ImportError:
    print("Optional SHAP skipped. Install with: python -m pip install shap")
try:
    from lime.lime_tabular import LimeTabularExplainer
    lime_explainer = LimeTabularExplainer(X_explain, feature_names=[f"f{i}" for i in range(5)],
                                           class_names=["no", "yes"], mode="classification")
    local_explanation = lime_explainer.explain_instance(X_explain[0], explain_model.predict_proba)
    print("LIME local explanation terms:", len(local_explanation.as_list()))
except ImportError:
    print("Optional LIME skipped. Install with: python -m pip install lime")


## 34. Production project blueprint: customer churn — [PRODUCTION CORE]

Suggested structure:

```text
customer_churn_ml/
  data/{raw,processed}/  notebooks/  src/
  src/{data_ingestion,data_validation,data_cleaning,feature_engineering,train,evaluate,predict}.py
  models/  tests/  api/app.py  monitoring/  Dockerfile  requirements.txt  dvc.yaml  README.md
```

Sequence: define target/cost/time; collect and version raw data; EDA and schema validation; clean; leakage-safe train/validation/test split; build ColumnTransformer; engineer/select only prediction-time features; compare logistic baseline, random forest, boosting; choose metric and tune; log MLflow; version data/pipeline with DVC; save complete pipeline; expose API or batch scoring; Docker/Kubernetes only if operations require; CI gates; monitor data/model/service; retrain only with evidence; register/promote/rollback.


In [ ]:
project_gates = [
    "target and prediction time reviewed",
    "schema/range/missingness checks pass",
    "split strategy matches time and entity structure",
    "pipeline fits all learned transforms on training folds only",
    "baseline and candidate compared on business metric",
    "artifact, code, data, environment, and metrics versioned",
    "API/batch contract and failure behavior tested",
    "monitoring, alert thresholds, approval, and rollback rehearsed",
]
for step, gate in enumerate(project_gates, start=1):
    print(f"{step:02}. {gate}")


## 35. Experienced ML interview practice: 32 questions and answer anchors

Use a structured answer: **definition -> trade-off -> when to use -> failure mode -> production example/metric**. Tailor the details to a real project and be explicit about what you personally owned.

1. **End-to-end production ML pipeline?** Frame decision/target/time; ingest/version; validate/EDA; split; leakage-safe pipeline; baseline/CV/tuning; business evaluation; register/deploy; monitor/retrain/rollback.
2. **Ingestion vs preprocessing?** Ingestion moves/records source data reliably; preprocessing validates, cleans, transforms features for the model. Keep raw source immutable and transformations reproducible.
3. **Mean vs median vs KNN vs iterative imputation?** Distribution/outliers, feature relationships, missingness mechanism, sample size, compute, and train-fold-only fitting determine choice; compare with validation.
4. **Detect/handle outliers?** Domain bounds then robust/univariate/multivariate detection; investigate; correct, cap, transform, robust model, or retain with reason. Never delete mechanically.
5. **Standard vs MinMax vs RobustScaler?** Center/standard deviation; fixed range; median/IQR. Choose by algorithm, distribution, outliers, sparse-data constraints; fit on train only.
6. **OneHot vs Ordinal vs TargetEncoder?** Nominal expansion; known meaningful order; target-statistics for high cardinality with cross-fitting. Unknown categories and leakage matter.
7. **Feature leakage example?** Post-outcome field, future aggregate, or target-derived feature; enforce prediction-time availability and temporal backtests.
8. **Why split before scaler?** Test statistics leaking into fit makes holdout optimistic; pipeline inside CV prevents it.
9. **Random Forest vs XGBoost vs AdaBoost?** Parallel bagged trees; regularized gradient boosting; sequential reweighting/boosting. Compare validation, calibration, latency, explainability, tuning burden.
10. **Why can XGBoost win on tabular data?** Flexible regularized boosting, missing-value handling and interactions, but not universally; data/metric/tuning and deployment constraints decide.
11. **Bagging vs boosting?** Parallel bootstrap aggregation mainly reduces variance; sequential boosting corrects prior residual/errors, often reducing bias but can overfit/noise-sensitive.
12. **Class imbalance?** Define costs, stratified/group/time splits, class weights/resample only train folds, tune threshold, PR metrics and calibration.
13. **Why misleading accuracy?** Majority-class predictor can score highly while missing minority positives; inspect confusion matrix, recall/precision, PR-AUC and business cost.
14. **Precision vs recall?** Precision if false positives costly; recall if false negatives costly; quantify capacity and expected cost.
15. **CV / StratifiedKFold?** Repeated train/validation folds estimate variability; stratification preserves class proportion for classification.
16. **TimeSeriesSplit?** Respects order and avoids training on future to predict past; add gaps/embargo for label overlap.
17. **Grid vs random vs Optuna?** Exhaustive small grids, budgeted broad random search, adaptive search; all use training CV and a final untouched test.
18. **Pipeline / ColumnTransformer?** Encapsulate column-specific transforms and estimator; consistent train/serve; fold-safe CV and fewer leakage bugs.
19. **What does MLflow track?** Runs, parameters, metrics, artifacts, models/registry; secure tracking and durable artifact storage.
20. **DVC solves?** Version large data/model artifacts and reproducible dependency pipelines alongside Git code.
21. **Deploy with FastAPI?** Versioned Pydantic contract, load full pipeline once, health/predict endpoints, errors/auth/timeouts/logging, container, integration tests.
22. **Batch vs real-time?** Batch optimizes throughput/cost and scheduled freshness; real-time gives low-latency per request but needs high availability and strict serving contracts.
23. **Why Docker?** Reproducible runtime/dependencies and portable artifact; pin, scan, non-root, no secrets, health checks.
24. **Pod/Deployment/Service/Ingress/HPA?** Workload unit; replica/rollout controller; stable networking; external HTTP routing; automatic horizontal scaling.
25. **ML CI/CD tests?** Unit/integration, schema/data, leakage, reproducibility, metric/fairness gates, API contract, dependency/image scan, smoke/rollback.
26. **Data vs concept drift?** Feature distribution changes vs P(y|X) relation changes; labels usually needed to verify quality impact.
27. **Monitor when labels are late?** Leading input/prediction/service metrics now; delayed outcome metrics when labels mature; cohort/label-lag-aware dashboards.
28. **When retrain?** Evidence of degradation/drift or scheduled business need, after data quality checks and representative labels; not every drift alert.
29. **Champion/challenger?** Same criteria/data, candidate shadow/canary, predeclared thresholds and governance before promotion.
30. **Rollback bad release?** Trigger on SLO/quality breach; restore immutable previous model/image/route; preserve incident evidence and test rollback.
31. **Explain one prediction with SHAP/LIME?** Define local vs global, background/perturbation, feature attribution and limitations; not causal; validate pipeline consistency.
32. **Reproducibility across train/serve?** Persist full pipeline, feature schema/code/data/environment/params/version; contract tests and same preprocessing artifact.


## 36. Study order and final production checklist

1. Python, pandas, NumPy, EDA, probability/statistics.
2. Data contracts, cleaning, missingness, outliers.
3. Encoding, scaling, transformations, feature engineering.
4. Regression/classification baselines, trees and ensembles.
5. Metrics, validation, leakage, imbalance, calibration.
6. Pipelines, feature selection, tuning and interpretation.
7. Serialization, MLflow/DVC, reproducibility.
8. Batch/API serving, Docker, cloud/Kubernetes fundamentals.
9. CI/CD, security, monitoring, drift, retraining, registry, rollback.
10. Explainability, fairness, governance, cost/latency trade-offs.

For each method, practice explaining what it is, why/when to use it, key parameters, assumptions, failure modes, alternatives, leakage risks, production implementation, expected output, and how success is measured.

**Production checklist:** prediction-time feature availability; correct temporal/group split; train-fold-only fit; versioned schema/pipeline/data/code/environment; representative metric and threshold; input/output validation; safe artifact handling; observability; alert/runbook; approval; rollback drill.
